# BKinD on JHMDB (split 1, clean) — Colab

Trains BKinD on clean JHMDB and scores it (Step 1 of the experiment plan).

**Before running**, put these files in Google Drive under `MyDrive/noisykeypoints/data/`:
- `JHMDB_rgb.zip` (frames, ~390 MB)
- `joint_positions.zip` (~14 MB)
- `splits.zip`

Checkpoints, logs and results are saved to `MyDrive/noisykeypoints/runs/`, so a disconnected session can resume.
**If Colab disconnects:** reconnect and run all cells again — training continues from the last saved epoch.

Use a GPU runtime: *Runtime → Change runtime type → GPU*.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# ---- Settings ----
REPO_URL = 'https://github.com/sleekEagle/noisykeypoints.git'
RUN_NAME = 'JHMDB_split1_clean'

from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/noisykeypoints'
DATA_ZIPS = f'{DRIVE}/data'
RUN_DIR = f'{DRIVE}/runs/{RUN_NAME}'
import os
os.makedirs(RUN_DIR, exist_ok=True)
for f in ['JHMDB_rgb.zip', 'joint_positions.zip', 'splits.zip']:
    assert os.path.exists(f'{DATA_ZIPS}/{f}'), f'missing {DATA_ZIPS}/{f}'
print('Drive OK, run folder:', RUN_DIR)

In [ ]:
# Get the code. For a private repo, add a GitHub token named GITHUB_TOKEN in Colab's
# Secrets panel (key icon on the left) with read access to the repo.
import os, subprocess
url = REPO_URL
try:
    from google.colab import userdata
    tok = userdata.get('GITHUB_TOKEN')
    url = REPO_URL.replace('https://', f'https://{tok}@')
except Exception:
    pass
if not os.path.exists('/content/noisykeypoints'):
    subprocess.run(['git', 'clone', '-q', url, '/content/noisykeypoints'], check=True)
else:
    subprocess.run(['git', '-C', '/content/noisykeypoints', 'pull', '-q'], check=True)
!git -C /content/noisykeypoints log -1 --format='code version: %h %s'

In [ ]:
# Colab already has torch, torchvision, scipy, sklearn, opencv, h5py, pyyaml, pandas, seaborn.
!pip install -q piq easydict

In [ ]:
# Copy data to the local disk (much faster than reading from Drive) and build BKinD's layout.
import os
if not os.path.exists('/content/JHMDB_bkind/split1/clips.json'):
    !mkdir -p /content/JHMDB /content/tmp
    !cp {DATA_ZIPS}/JHMDB_rgb.zip {DATA_ZIPS}/joint_positions.zip {DATA_ZIPS}/splits.zip /content/tmp/
    !unzip -q /content/tmp/JHMDB_rgb.zip -d /content/tmp && mv /content/tmp/JHMDB_rgb /content/JHMDB/frames
    !unzip -q /content/tmp/joint_positions.zip -x '*.AppleDouble*' -d /content/JHMDB
    !unzip -q /content/tmp/splits.zip -d /content/JHMDB
    !rm -rf /content/tmp
    !python /content/noisykeypoints/scripts/prepare_jhmdb_bkind.py --jhmdb /content/JHMDB --out /content/JHMDB_bkind --split 1
else:
    print('data already prepared')

In [ ]:
# Make a Colab copy of the JHMDB config: local data path, checkpoints on Drive,
# and batch 32 (BKinD's setting at 128 px) when the GPU has enough memory.
import yaml, torch
BK = '/content/noisykeypoints/third_party/BKinD'
cfg = yaml.safe_load(open(f'{BK}/config/JHMDB.yaml'))
gpu_gb = torch.cuda.get_device_properties(0).total_memory / 2**30
cfg.update(data='/content/JHMDB_bkind/split1', checkpoint=RUN_DIR,
           batch_size=32 if gpu_gb >= 14 else 16, workers=2)
yaml.safe_dump(cfg, open(f'{BK}/config/JHMDB_colab.yaml', 'w'))
print(f'GPU memory {gpu_gb:.1f} GB -> batch_size {cfg["batch_size"]}')

In [ ]:
# Train. Resumes automatically from the last saved epoch if a checkpoint is on Drive.
import os
ckpt = f'{RUN_DIR}/checkpoint.pth.tar'
resume = f'--resume {ckpt}' if os.path.exists(ckpt) else ''
print('resuming from', ckpt) if resume else print('starting fresh')
%cd {BK}
!PYTHONUNBUFFERED=1 python train_video.py --config config/JHMDB_colab.yaml {resume} 2>&1 | grep --line-buffered -v "Warning" | tee -a {RUN_DIR}/train_log.txt

In [ ]:
# Training curve: learning rate, train loss, val loss per epoch.
!cat {RUN_DIR}/log.txt

In [ ]:
# Score the trained model on the val (official test) clips.
%cd /content/noisykeypoints
!python scripts/eval_bkind_jhmdb.py --checkpoint {RUN_DIR}/checkpoint.pth.tar \
    --data /content/JHMDB_bkind/split1 --jhmdb /content/JHMDB --out {RUN_DIR}/eval

In [ ]:
from IPython.display import Image, display
display(Image(f'{RUN_DIR}/eval/vis_val.png'))